# Dense-CLIP Multilingual Grounding — Analysis Notebook
## BDD100K Visual Grounding · 2 CLIP backbones × 13 Languages × 11 Concepts
### Companion analysis to the dense-CLIP follow-up paper

This notebook reproduces every table, figure and statistic for the dense-CLIP
follow-up paper from the per-language and per-concept summary files written by
the `language_conditioned_maps_decurto` pipeline. It complements the parent
notebook (`vlm_multilingual_energy_analysis_v4_200.ipynb`) by taking the same
210-image frozen subset, the same 13 languages and the same low-resource tag
set `{ar, eu, lb}` and analysing dense visual–text alignment instead of
generative VLM scoring.

**CLIP backbones evaluated:**

| Short name | OpenCLIP ID | Pretrained | Visual params |
|---|---|---|---|
| XLM-R base + ViT-B/32  | `xlm-roberta-base-ViT-B-32`   | `laion5b_s13b_b90k`        | ~87 M  |
| XLM-R large + ViT-H/14 | `xlm-roberta-large-ViT-H-14`  | `frozen_laion5b_s13b_b90k` | ~632 M |

**Research questions:**

1. **RQ1 — Persistence under shared visual encoder.** Does the multilingual
   penalty persist when the visual encoder is *identical* across all 13
   languages and only the text branch changes?
2. **RQ2 — Scale dependence.** Does increasing the text encoder size (and
   visual encoder capacity) close the low-resource gap?
3. **RQ3 — Mechanism.** Is the failure mode *signal collapse* (target
   languages produce weak peak similarity) or *spatial misalignment* (peak
   is preserved but the location of activation differs)?

**Comparison to parent paper:**

The energy figures from this probe are directly comparable to the AI Energy
Score figures of the parent paper (same NVML 10 Hz sampling, same Wh / 1K
queries unit). Cell 9 quantifies the energy advantage of dense-CLIP grounding
relative to autoregressive VLM querying.

**Inputs expected:** the contents of the two result directories from the
student's pipeline (one per backbone). Place both directories side-by-side and
point `INPUT_DIR` to their parent.


In [1]:
# ============================================================
# 0) Setup — imports, config, helpers
# ============================================================

!pip -q install pandas numpy matplotlib scipy

import os, re, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy import stats

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 200,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

# ── Paths ────────────────────────────────────────────────────────────────────
# Each backbone result directory is expected directly under INPUT_DIR.
# Adjust if your student's pipeline writes elsewhere.
INPUT_DIR = "."
OUTDIR    = "clip_multilingual_analysis"
BASELINE  = "en"

# Backbones evaluated. The keys are the result-directory names; the values are
# (display name, visual-encoder param count in M) — used for tables/legends.
BACKBONES = {
    "dense_clip_xlmr_base":  ("XLM-R base + ViT-B/32",  87),
    "dense_clip_xlmr_large": ("XLM-R large + ViT-H/14", 632),
}

os.makedirs(OUTDIR, exist_ok=True)

# ── Language metadata (same as parent paper) ─────────────────────────────────
LANG_NAMES = {
    "ar": "Arabic",    "ca": "Catalan",      "de": "German",
    "en": "English",   "es": "Spanish",      "eu": "Basque",
    "fr": "French",    "it": "Italian",      "lb": "Luxembourgish",
    "pt": "Portuguese","ru": "Russian",      "zh-CN": "Chinese (Simp.)",
    "zh-TW": "Chinese (Trad.)",
}

LOW_RESOURCE = {"ar", "eu", "lb"}

# Display order: English first, then Romance, then Germanic, then everything else
LANG_ORDER = ["en","es","fr","it","pt","ca","de","lb","ru","ar","eu","zh-CN","zh-TW"]

# Family colours — reused throughout the notebook for consistency with the
# pipeline's default figures.
FAMILY_COLOR = {
    "Germanic": "#1f77b4",
    "Romance":  "#ff7f0e",
    "Slavic":   "#2ca02c",
    "Sinitic":  "#d4a017",
    "Semitic":  "#d62728",
    "Isolate":  "#7e3fbf",
}

# Backbone colours
BACKBONE_COLOR = {
    "dense_clip_xlmr_base":  "#1f77b4",
    "dense_clip_xlmr_large": "#d62728",
}

# Concepts evaluated (BDD100K-relevant nouns) — order used in heatmaps
CONCEPT_ORDER = [
    "car", "truck", "bus", "person", "pedestrian", "traffic_light",
    "traffic_sign", "bicycle", "motorcycle", "road", "building",
]

# ── Utilities ────────────────────────────────────────────────────────────────
def savefig(name, tight=True):
    path = os.path.join(OUTDIR, name)
    if tight:
        plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print("Saved:", path)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


print("Setup complete — output directory:", OUTDIR)
print("Expected input directories:")
for k, (name, params) in BACKBONES.items():
    print(f"  {INPUT_DIR}/{k}/  ({name}, ~{params} M visual params)")


Setup complete — output directory: clip_multilingual_analysis
Expected input directories:
  ./dense_clip_xlmr_base/  (XLM-R base + ViT-B/32, ~87 M visual params)
  ./dense_clip_xlmr_large/  (XLM-R large + ViT-H/14, ~632 M visual params)


In [2]:
# ============================================================
# 1) Load both backbones — build tidy DataFrames
# ============================================================
#
# We build four tables:
#   df_lang     : 1 row per (backbone, language)            — from per_language_summary.json
#   df_conc     : 1 row per (backbone, language, concept)   — from per_concept_summary.json
#   df_pair     : 1 row per (backbone, image, concept, language) — JSONL, paired stats
#   df_energy   : 1 row per backbone                        — from energy.json
#
# JSONL is the heaviest object (~30k rows × 2 backbones); we load it once and
# cache. All later cells read from these four DataFrames.

rows_lang   = []
rows_conc   = []
rows_pair   = []
rows_energy = []

for backbone, (display_name, vis_params_M) in BACKBONES.items():
    bdir = os.path.join(INPUT_DIR, backbone)
    if not os.path.isdir(bdir):
        raise FileNotFoundError(
            f"Missing input directory: {bdir}\n"
            "Place the two result folders under INPUT_DIR before running."
        )

    # 1a · per-language summary
    summ = load_json(os.path.join(bdir, "per_language_summary.json"))
    for lang, d in summ.items():
        rows_lang.append({
            "backbone":     backbone,
            "display":      display_name,
            "vis_params_M": vis_params_M,
            "lang":         lang,
            "lang_name":    LANG_NAMES.get(lang, lang),
            "low_resource": lang in LOW_RESOURCE,
            **{k: v for k, v in d.items() if k != "image_idx"},
        })

    # 1b · per-concept × per-language summary  (concept → lang → metrics)
    pc = load_json(os.path.join(bdir, "per_concept_summary.json"))
    for concept, lang_dict in pc.items():
        for lang, d in lang_dict.items():
            rows_conc.append({
                "backbone":     backbone,
                "display":      display_name,
                "concept":      concept,
                "lang":         lang,
                "lang_name":    LANG_NAMES.get(lang, lang),
                "low_resource": lang in LOW_RESOURCE,
                **{k: v for k, v in d.items() if k != "image_idx"},
            })

    # 1c · per-sample paired records (for paired stats)
    jsonl = load_jsonl(os.path.join(bdir, "per_sample_records.jsonl"))
    for r in jsonl:
        rows_pair.append({"backbone": backbone, **r})

    # 1d · energy block
    e = load_json(os.path.join(bdir, "energy.json"))
    rows_energy.append({
        "backbone":          backbone,
        "display":           display_name,
        "vis_params_M":      vis_params_M,
        "duration_sec":      e["duration_sec"],
        "avg_watts":         e["avg_watts"],
        "max_watts":         e["max_watts"],
        "min_watts":         e["min_watts"],
        "total_wh":          e["total_wh"],
        "n_queries":         e["n_queries"],
        "wh_per_1k_queries": e["wh_per_1k_queries"],
        "images_evaluated":  e["images_evaluated"],
    })

df_lang   = pd.DataFrame(rows_lang)
df_conc   = pd.DataFrame(rows_conc)
df_pair   = pd.DataFrame(rows_pair)
df_energy = pd.DataFrame(rows_energy)

# Add language family by joining with the per-language CSV from the pipeline
csv_paths = {b: os.path.join(INPUT_DIR, b, "figures", "per_language_table.csv")
             for b in BACKBONES}
fam_df = (pd.read_csv(next(iter(csv_paths.values())))
            [["language", "family"]]
            .rename(columns={"language": "lang"}))
df_lang = df_lang.merge(fam_df, on="lang", how="left")
df_conc = df_conc.merge(fam_df, on="lang", how="left")

# Save tidy CSVs (handy for the paper's supplementary material)
df_lang.to_csv(os.path.join(OUTDIR, "df_lang.csv"), index=False)
df_conc.to_csv(os.path.join(OUTDIR, "df_conc.csv"), index=False)
df_energy.to_csv(os.path.join(OUTDIR, "df_energy.csv"), index=False)

print(f"df_lang   : {df_lang.shape[0]} rows  ({len(BACKBONES)} backbones × 13 languages)")
print(f"df_conc   : {df_conc.shape[0]} rows  ({len(BACKBONES)} × 13 × 11 concepts)")
print(f"df_pair   : {df_pair.shape[0]} rows  (per-sample paired records)")
print(f"df_energy : {df_energy.shape[0]} rows  ({len(BACKBONES)} backbones)")
print()
print("Energy summary:")
display(df_energy[["display", "duration_sec", "avg_watts",
                   "wh_per_1k_queries", "total_wh"]].round(3).to_string(index=False))


df_lang   : 26 rows  (2 backbones × 13 languages)
df_conc   : 286 rows  (2 × 13 × 11 concepts)
df_pair   : 60060 rows  (per-sample paired records)
df_energy : 2 rows  (2 backbones)

Energy summary:


'               display  duration_sec  avg_watts  wh_per_1k_queries  total_wh\n XLM-R base + ViT-B/32      4227.374     98.912              3.868   116.150\nXLM-R large + ViT-H/14      3657.317     99.273              3.358   100.854'

In [3]:
# ============================================================
# 2) Descriptive statistics
# ============================================================

# Primary metrics carried through the paper.
PRIMARY_METRICS = ["iou_cluster_mask", "iou_pct_95",
                   "spearman", "peak_lang", "peak_ratio_lang_over_ref",
                   "center_dist_symmetric"]

# ── 2a · Per-backbone × per-language wide table (paper-ready) ────────────────
desc_lang = (df_lang
             .pivot_table(index="lang",
                          columns="backbone",
                          values=PRIMARY_METRICS)
             .reindex([l for l in LANG_ORDER if l in df_lang["lang"].unique()]))
desc_lang.to_csv(os.path.join(OUTDIR, "table_per_language_per_backbone.csv"))

# ── 2b · Per-backbone aggregates (HR / LR / overall) ─────────────────────────
agg_rows = []
for backbone, gb in df_lang.groupby("backbone"):
    non_en = gb[gb["lang"] != "en"]
    hr     = non_en[~non_en["low_resource"]]
    lr     = non_en[ non_en["low_resource"]]
    agg_rows.append({
        "backbone":   backbone,
        "display":    gb["display"].iloc[0],
        "iou_HR":     hr["iou_cluster_mask"].mean(),
        "iou_LR":     lr["iou_cluster_mask"].mean(),
        "iou_gap":    hr["iou_cluster_mask"].mean() - lr["iou_cluster_mask"].mean(),
        "spearman_HR":  hr["spearman"].mean(),
        "spearman_LR":  lr["spearman"].mean(),
        "spearman_gap": hr["spearman"].mean() - lr["spearman"].mean(),
        "peak_HR":      hr["peak_lang"].mean(),
        "peak_LR":      lr["peak_lang"].mean(),
        "peak_gap":     hr["peak_lang"].mean() - lr["peak_lang"].mean(),
    })
df_agg = pd.DataFrame(agg_rows)
df_agg.to_csv(os.path.join(OUTDIR, "table_HR_vs_LR_per_backbone.csv"), index=False)

print("Per-backbone HR vs LR summary (non-English languages):")
display(df_agg.round(4).to_string(index=False))
print()
print("Saved:", os.path.join(OUTDIR, "table_per_language_per_backbone.csv"))
print("Saved:", os.path.join(OUTDIR, "table_HR_vs_LR_per_backbone.csv"))


Per-backbone HR vs LR summary (non-English languages):


'             backbone                display  iou_HR  iou_LR  iou_gap  spearman_HR  spearman_LR  spearman_gap  peak_HR  peak_LR  peak_gap\n dense_clip_xlmr_base  XLM-R base + ViT-B/32  0.6123  0.4978   0.1145       0.8762       0.7274        0.1489   0.1975   0.1956    0.0018\ndense_clip_xlmr_large XLM-R large + ViT-H/14  0.6076  0.4647   0.1428       0.8903       0.6945        0.1958   0.1667   0.1592    0.0075'


Saved: clip_multilingual_analysis/table_per_language_per_backbone.csv
Saved: clip_multilingual_analysis/table_HR_vs_LR_per_backbone.csv


In [4]:
# ============================================================
# 3) Statistical analysis  —  paired tests across both backbones
# ============================================================
#
# The paired statistical machinery follows the parent paper:
#   - Friedman across non-English languages (paired by image, concept)
#   - Wilcoxon HR > LR (one-sided, paired by image, concept)
#   - Mann-Whitney per-LR-language vs HR pool (one-sided)
#
# We run each test independently on each backbone and keep both columns of
# results so the final paper can report the persistence under scale.

PAIRED_METRICS = ["iou_cluster_mask", "iou_pct_95", "spearman", "peak_lang",
                  "peak_ratio_lang_over_ref"]

def friedman_across_langs(df, metric, ref="en"):
    """Friedman test across non-reference languages, blocks = (image, concept)."""
    df = df[df["language"] != ref]
    pivot = (df.pivot_table(index=["image_idx", "concept"],
                            columns="language",
                            values=metric)
              .dropna())
    if pivot.shape[1] < 3 or pivot.shape[0] < 5:
        return np.nan, np.nan, pivot.shape[0]
    stat, p = stats.friedmanchisquare(*[pivot[c].values for c in pivot.columns])
    return float(stat), float(p), int(pivot.shape[0])


def wilcoxon_hr_vs_lr(df, metric, ref="en"):
    """Wilcoxon HR > LR: per (image, concept) compute mean over HR / LR langs."""
    df = df[df["language"] != ref].copy()
    df["is_lr"] = df["language"].isin(LOW_RESOURCE)
    grp = (df.groupby(["image_idx", "concept", "is_lr"])[metric]
             .mean()
             .unstack("is_lr"))
    grp.columns = ["hr", "lr"]
    grp = grp.dropna()
    if grp.shape[0] < 5:
        return np.nan, np.nan, np.nan, np.nan
    stat, p = stats.wilcoxon(grp["hr"], grp["lr"], alternative="greater")
    return float(stat), float(p), float((grp["hr"] - grp["lr"]).mean()), int(grp.shape[0])


def mw_per_lr(df, metric, ref="en"):
    """Mann–Whitney each LR language vs HR pool, one-sided (HR > LR)."""
    df = df[df["language"] != ref].copy()
    out = {}
    hr_vals = df[~df["language"].isin(LOW_RESOURCE)][metric].dropna().values
    for lr in sorted(LOW_RESOURCE):
        lr_vals = df[df["language"] == lr][metric].dropna().values
        if len(hr_vals) < 5 or len(lr_vals) < 5:
            out[lr] = (np.nan, np.nan)
            continue
        stat, p = stats.mannwhitneyu(hr_vals, lr_vals, alternative="greater")
        out[lr] = (float(stat), float(p))
    return out


stat_rows = []
for backbone, gb in df_pair.groupby("backbone"):
    for m in PAIRED_METRICS:
        fr_s, fr_p, fr_n = friedman_across_langs(gb, m)
        wx_s, wx_p, wx_d, wx_n = wilcoxon_hr_vs_lr(gb, m)
        mw = mw_per_lr(gb, m)
        stat_rows.append({
            "backbone":         backbone,
            "metric":           m,
            "friedman_stat":    fr_s,
            "friedman_p":       fr_p,
            "friedman_n":       fr_n,
            "wilcoxon_stat":    wx_s,
            "wilcoxon_p":       wx_p,
            "wilcoxon_meandif": wx_d,
            "wilcoxon_n":       wx_n,
            "mw_ar_p":          mw["ar"][1],
            "mw_eu_p":          mw["eu"][1],
            "mw_lb_p":          mw["lb"][1],
        })

df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(os.path.join(OUTDIR, "table_stat_tests.csv"), index=False)

# Pretty print
print("Friedman across non-English languages, blocks=(image,concept):\n")
sub = df_stats.assign(p_str=lambda d: d["friedman_p"].apply(
    lambda p: f"{p:.2e}" if not np.isnan(p) else "—"))
display(sub.pivot_table(index="metric", columns="backbone", values="friedman_stat")
          .round(1).to_string())
print()
print("Wilcoxon HR > LR (paired, mean diff and p):\n")
display(df_stats.assign(
    summary=lambda d: d.apply(
        lambda r: f"Δ={r['wilcoxon_meandif']:+.4f}, p={r['wilcoxon_p']:.2e}",
        axis=1))
    .pivot_table(index="metric", columns="backbone", values="summary",
                 aggfunc="first")
    .to_string())
print()
print("Saved:", os.path.join(OUTDIR, "table_stat_tests.csv"))


Friedman across non-English languages, blocks=(image,concept):



'backbone                  dense_clip_xlmr_base  dense_clip_xlmr_large\nmetric                                                               \niou_cluster_mask                        8065.8                 9478.4\niou_pct_95                              5326.5                 7894.7\npeak_lang                               7794.4                 3046.2\npeak_ratio_lang_over_ref                7794.4                 3046.2\nspearman                               11803.1                11152.1'


Wilcoxon HR > LR (paired, mean diff and p):



'backbone                    dense_clip_xlmr_base   dense_clip_xlmr_large\nmetric                                                                  \niou_cluster_mask           Δ=+0.1145, p=0.00e+00   Δ=+0.1428, p=0.00e+00\niou_pct_95                Δ=+0.1254, p=3.64e-278   Δ=+0.1700, p=0.00e+00\npeak_lang                  Δ=+0.0018, p=4.72e-02  Δ=+0.0075, p=2.18e-124\npeak_ratio_lang_over_ref   Δ=+0.0049, p=9.79e-02  Δ=+0.0370, p=5.30e-111\nspearman                   Δ=+0.1489, p=0.00e+00   Δ=+0.1958, p=0.00e+00'


Saved: clip_multilingual_analysis/table_stat_tests.csv


In [5]:
# ============================================================
# 4) Figure 1 — Per-language cross-language IoU, both backbones
#    Shows that the multilingual penalty persists at both scales.
# ============================================================

ordered = [l for l in LANG_ORDER if l in df_lang["lang"].unique()]
fig, ax = plt.subplots(figsize=(11, 4.5))

bb_keys = list(BACKBONES.keys())
bar_w   = 0.4
x       = np.arange(len(ordered))

for i, bb in enumerate(bb_keys):
    vals  = (df_lang[df_lang["backbone"] == bb]
                .set_index("lang")
                .loc[ordered, "iou_cluster_mask"]
                .values)
    stds  = (df_lang[df_lang["backbone"] == bb]
                .set_index("lang")
                .loc[ordered, "iou_cluster_mask"]
                .values * 0)  # SE not in summary; CSV has std (per-sample), not SE
    bars = ax.bar(x + (i - 0.5) * bar_w, vals, width=bar_w,
                  color=BACKBONE_COLOR[bb], edgecolor="black", linewidth=0.5,
                  label=BACKBONES[bb][0])
    for b, v, lang in zip(bars, vals, ordered):
        if lang in LOW_RESOURCE:
            ax.text(b.get_x() + b.get_width() / 2, v + 0.015, "⚠",
                    ha="center", va="bottom", fontsize=9, color="black")

ax.set_xticks(x)
ax.set_xticklabels([f"{LANG_NAMES.get(l,l)}\n({l})" for l in ordered],
                   rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Cluster-mask IoU vs English")
ax.set_title("Per-language cross-language agreement — both CLIP backbones\n"
             "⚠ = low-resource (Arabic, Basque, Luxembourgish)")
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color="grey", lw=0.5, ls="--")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="lower left", frameon=True)

savefig("fig1_per_language_iou_both_backbones.png")


Saved: clip_multilingual_analysis/fig1_per_language_iou_both_backbones.png


In [6]:
# ============================================================
# 5) Figure 2 — HR vs LR gap across metrics and backbones
#    "Forest plot" — does the gap close, hold, or widen with scale?
# ============================================================

metrics_plot = ["iou_cluster_mask", "iou_pct_95", "spearman", "peak_lang"]
metric_label = {
    "iou_cluster_mask": "Cluster-mask IoU vs EN",
    "iou_pct_95":       "Top-5% IoU vs EN",
    "spearman":         "Spearman ρ vs EN",
    "peak_lang":        "Mean peak similarity",
}

fig, ax = plt.subplots(figsize=(9, 5))
y_offsets = {
    "dense_clip_xlmr_base":  +0.18,
    "dense_clip_xlmr_large": -0.18,
}

for backbone in BACKBONES:
    sub = df_agg[df_agg["backbone"] == backbone].iloc[0]
    for j, m in enumerate(metrics_plot):
        if m == "peak_lang":
            hr_v = sub["peak_HR"]
            lr_v = sub["peak_LR"]
        elif m == "spearman":
            hr_v = sub["spearman_HR"]
            lr_v = sub["spearman_LR"]
        elif m == "iou_pct_95":
            hr_v = (df_lang[(df_lang["backbone"]==backbone) & (df_lang["lang"]!="en") &
                            (~df_lang["low_resource"])]["iou_pct_95"].mean())
            lr_v = (df_lang[(df_lang["backbone"]==backbone) & (df_lang["lang"]!="en") &
                            ( df_lang["low_resource"])]["iou_pct_95"].mean())
        else:
            hr_v = sub["iou_HR"]
            lr_v = sub["iou_LR"]
        y = j + y_offsets[backbone]
        # Draw HR (open circle) and LR (filled square) with a connecting line
        ax.plot([lr_v, hr_v], [y, y], "-",
                color=BACKBONE_COLOR[backbone], linewidth=1.5)
        ax.plot(hr_v, y, "o", color=BACKBONE_COLOR[backbone], markersize=8,
                markeredgecolor="black", markeredgewidth=0.6,
                label=(BACKBONES[backbone][0] + " — HR" if j == 0 else None))
        ax.plot(lr_v, y, "s", color=BACKBONE_COLOR[backbone], markersize=8,
                markeredgecolor="black", markeredgewidth=0.6, alpha=0.6,
                label=(BACKBONES[backbone][0] + " — LR" if j == 0 else None))
        ax.text(hr_v, y + 0.06, f"{hr_v:.3f}", ha="center", va="bottom", fontsize=8)
        ax.text(lr_v, y - 0.06, f"{lr_v:.3f}", ha="center", va="top",   fontsize=8)

ax.set_yticks(range(len(metrics_plot)))
ax.set_yticklabels([metric_label[m] for m in metrics_plot])
ax.invert_yaxis()
ax.set_xlabel("Metric value (higher = better cross-language agreement)")
ax.set_title("HR vs LR gap across metrics and backbones\n"
             "○ HR mean    ■ LR mean    line = gap")
ax.grid(axis="x", alpha=0.25)
# Custom legend (we only added handles for the first metric)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc="lower left", frameon=True, fontsize=8)

savefig("fig2_hr_vs_lr_forest.png")

print("HR – LR gap by metric and backbone:")
for backbone in BACKBONES:
    sub = df_agg[df_agg["backbone"] == backbone].iloc[0]
    print(f"  {BACKBONES[backbone][0]:30s}  IoU gap = {sub['iou_gap']:+.4f}, "
          f"Spearman gap = {sub['spearman_gap']:+.4f}")


Saved: clip_multilingual_analysis/fig2_hr_vs_lr_forest.png
HR – LR gap by metric and backbone:
  XLM-R base + ViT-B/32           IoU gap = +0.1145, Spearman gap = +0.1489
  XLM-R large + ViT-H/14          IoU gap = +0.1428, Spearman gap = +0.1958


In [7]:
# ============================================================
# 6) Figure 3 — Per-language Δ-IoU under scaling   (RQ2)
#    For each non-English language: IoU(large) − IoU(base).
#    Negative bars show languages that *get worse* at scale.
# ============================================================

base  = (df_lang[df_lang["backbone"] == "dense_clip_xlmr_base"]
            .set_index("lang"))
large = (df_lang[df_lang["backbone"] == "dense_clip_xlmr_large"]
            .set_index("lang"))

ordered_no_en = [l for l in LANG_ORDER if l != "en" and l in base.index]

deltas = pd.DataFrame({
    "lang":         ordered_no_en,
    "lang_name":    [LANG_NAMES[l] for l in ordered_no_en],
    "family":       [base.loc[l, "family"] for l in ordered_no_en],
    "low_resource": [l in LOW_RESOURCE for l in ordered_no_en],
    "iou_base":     [base.loc[l, "iou_cluster_mask"] for l in ordered_no_en],
    "iou_large":    [large.loc[l, "iou_cluster_mask"] for l in ordered_no_en],
}).assign(delta=lambda d: d["iou_large"] - d["iou_base"])

deltas.to_csv(os.path.join(OUTDIR, "table_delta_iou_large_minus_base.csv"),
              index=False)

fig, ax = plt.subplots(figsize=(11, 4.2))
colors = [FAMILY_COLOR.get(f, "grey") for f in deltas["family"]]
bars = ax.bar(range(len(deltas)), deltas["delta"], color=colors,
              edgecolor="black", linewidth=0.5)

# Mark low-resource languages with a hatched border
for b, lr in zip(bars, deltas["low_resource"]):
    if lr:
        b.set_hatch("///")
        b.set_edgecolor("black")
        b.set_linewidth(1.0)

ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(range(len(deltas)))
ax.set_xticklabels([f"{r.lang_name}\n({r.lang})" for _, r in deltas.iterrows()],
                   rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Δ IoU = IoU(large) − IoU(base)")
ax.set_title("Effect of scaling text + visual encoder (~7× visual params) on "
             "cross-language agreement\n"
             "Hatched bars = low-resource languages; positive = scale closes "
             "the gap, negative = scale widens it")
ax.grid(axis="y", alpha=0.25)

# Family legend
from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=c, edgecolor="black", label=f)
                for f, c in FAMILY_COLOR.items()]
legend_elems.append(Patch(facecolor="white", edgecolor="black",
                          hatch="///", label="Low-resource"))
ax.legend(handles=legend_elems, loc="lower right", fontsize=8, frameon=True)

savefig("fig3_delta_iou_scale.png")

print("Δ IoU (large − base), low-resource only:")
display(deltas[deltas["low_resource"]][["lang", "iou_base", "iou_large", "delta"]]
        .round(4).to_string(index=False))
print()
print("Δ IoU (large − base), high-resource only:")
display(deltas[~deltas["low_resource"]][["lang", "iou_base", "iou_large", "delta"]]
        .round(4).to_string(index=False))


Saved: clip_multilingual_analysis/fig3_delta_iou_scale.png
Δ IoU (large − base), low-resource only:


'lang  iou_base  iou_large   delta\n  lb    0.4640     0.3875 -0.0764\n  ar    0.5300     0.5628  0.0328\n  eu    0.4994     0.4438 -0.0555'


Δ IoU (large − base), high-resource only:


' lang  iou_base  iou_large   delta\n   es    0.6535     0.6512 -0.0023\n   fr    0.6450     0.6237 -0.0213\n   it    0.6517     0.6152 -0.0365\n   pt    0.6476     0.6270 -0.0206\n   ca    0.6253     0.6102 -0.0151\n   de    0.6357     0.6210 -0.0147\n   ru    0.5552     0.5731  0.0179\nzh-CN    0.5334     0.5728  0.0394\nzh-TW    0.5630     0.5739  0.0109'

In [8]:
# ============================================================
# 7) Figure 4 — Mechanism: signal collapse or spatial misalignment?  (RQ3)
#
# Signal-collapse hypothesis  : low-resource languages produce a weaker
#                               peak similarity than English on the same image.
# Spatial-misalignment        : peak similarity is preserved (or even higher),
#                               but cluster-mask IoU drops because activation
#                               is in the *wrong* region.
#
# Plot peak_ratio_lang_over_ref against cluster-mask IoU per language and
# backbone. A signal-collapse failure mode lands in the lower-left; a pure
# spatial-misalignment failure mode lands in the lower-right or above 1.0.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, backbone in zip(axes, BACKBONES):
    sub = (df_lang[(df_lang["backbone"] == backbone) & (df_lang["lang"] != "en")]
              .copy())
    for _, r in sub.iterrows():
        c = FAMILY_COLOR.get(r["family"], "grey")
        marker = "D" if r["low_resource"] else "o"
        size = 130 if r["low_resource"] else 80
        ax.scatter(r["peak_ratio_lang_over_ref"], r["iou_cluster_mask"],
                   c=c, marker=marker, s=size, edgecolor="black", linewidth=0.7,
                   zorder=3)
        ax.annotate(r["lang"],
                    xy=(r["peak_ratio_lang_over_ref"], r["iou_cluster_mask"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=8)

    ax.axvline(1.0, color="grey", linewidth=0.5, ls="--")
    ax.set_title(f"{BACKBONES[backbone][0]}")
    ax.set_xlabel("Peak similarity ratio (target / English)")
    ax.grid(alpha=0.25)
    # Annotate quadrants
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.text(0.02, 0.97, "spatial misalignment\n(peak preserved, IoU low)",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.7))
    ax.text(0.55, 0.05, "signal-strength gain\n(non-EN beats EN)",
            transform=ax.transAxes, fontsize=8, va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.7))

axes[0].set_ylabel("Cluster-mask IoU vs English")

# Family legend
legend_elems = [plt.Line2D([0], [0], marker="o", color="w",
                            markerfacecolor=c, markeredgecolor="black", label=f)
                for f, c in FAMILY_COLOR.items()]
legend_elems += [
    plt.Line2D([0], [0], marker="D", color="w", markerfacecolor="white",
               markeredgecolor="black", label="Low-resource"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="white",
               markeredgecolor="black", label="High-resource"),
]
fig.legend(handles=legend_elems, loc="upper center",
           bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=8, frameon=True)

plt.suptitle("Mechanism diagnostic: signal collapse vs spatial misalignment\n"
             "(peak ratio ≈ 1 with low IoU ⇒ activation is in the wrong region)",
             y=1.02)
savefig("fig4_signal_vs_spatial.png")

# Numerical decomposition: per low-resource language, show peak ratio and IoU
print("Mechanism summary (low-resource languages only):\n")
for backbone in BACKBONES:
    print(f"  {BACKBONES[backbone][0]}")
    sub = (df_lang[(df_lang["backbone"] == backbone) & df_lang["low_resource"]]
                [["lang", "peak_ratio_lang_over_ref", "iou_cluster_mask",
                  "spearman", "center_dist_symmetric"]])
    print(sub.round(4).to_string(index=False))
    print()


Saved: clip_multilingual_analysis/fig4_signal_vs_spatial.png
Mechanism summary (low-resource languages only):

  XLM-R base + ViT-B/32
lang  peak_ratio_lang_over_ref  iou_cluster_mask  spearman  center_dist_symmetric
  ar                    1.0729            0.5300    0.7970                32.5062
  eu                    0.9286            0.4994    0.7177                34.4876
  lb                    0.9309            0.4640    0.6673                37.4831

  XLM-R large + ViT-H/14
lang  peak_ratio_lang_over_ref  iou_cluster_mask  spearman  center_dist_symmetric
  ar                    0.9368            0.5628    0.8556                14.6368
  eu                    0.9154            0.4438    0.6508                17.1630
  lb                    0.8742            0.3875    0.5769                18.5876



In [9]:
# ============================================================
# 8) Figure 5 — Concept-localised collapse  (per-language × per-concept IoU)
# ============================================================

def heatmap_for_backbone(backbone, ax, cbar=True):
    sub = (df_conc[df_conc["backbone"] == backbone]
              .pivot_table(index="lang", columns="concept",
                           values="iou_cluster_mask"))
    # Re-order rows/columns to match paper convention
    rows_in_order = [l for l in LANG_ORDER if l in sub.index]
    cols_in_order = [c for c in CONCEPT_ORDER if c in sub.columns]
    sub = sub.reindex(rows_in_order, axis=0).reindex(cols_in_order, axis=1)

    im = ax.imshow(sub.values, cmap="viridis", vmin=0.1, vmax=1.0,
                   aspect="auto")
    ax.set_xticks(range(sub.shape[1]))
    ax.set_xticklabels(sub.columns, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(sub.shape[0]))
    ax.set_yticklabels(sub.index, fontsize=9)
    for i in range(sub.shape[0]):
        for j in range(sub.shape[1]):
            v = sub.iloc[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=7,
                        color="white" if v < 0.5 else "black")
    # Highlight low-resource rows
    for i, lang in enumerate(sub.index):
        if lang in LOW_RESOURCE:
            ax.add_patch(plt.Rectangle((-0.5, i - 0.5), sub.shape[1], 1,
                                       fill=False, edgecolor="red",
                                       linewidth=1.5))
    ax.set_title(BACKBONES[backbone][0])
    if cbar:
        plt.colorbar(im, ax=ax, label="Cluster-mask IoU vs EN", shrink=0.8)
    return sub


fig, axes = plt.subplots(1, 2, figsize=(20, 6.5))
for ax, backbone in zip(axes, BACKBONES):
    _ = heatmap_for_backbone(backbone, ax)

plt.suptitle("Per-language × per-concept cluster-mask IoU vs English\n"
             "Red boxes mark the three low-resource languages")
savefig("fig5_concept_heatmaps_both_backbones.png")


Saved: clip_multilingual_analysis/fig5_concept_heatmaps_both_backbones.png


In [10]:
# ============================================================
# 9) Figure 6 — Low-resource concept profile
#    For each LR language, plot per-concept IoU under both backbones.
#    Shows the concept-specific signature of the failure mode.
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)

for ax, lang in zip(axes, ["ar", "eu", "lb"]):
    for bb in BACKBONES:
        sub = (df_conc[(df_conc["backbone"] == bb) & (df_conc["lang"] == lang)]
                  .set_index("concept")
                  .reindex(CONCEPT_ORDER)
                  ["iou_cluster_mask"])
        ax.plot(range(len(sub)), sub.values, "-o",
                color=BACKBONE_COLOR[bb], label=BACKBONES[bb][0],
                markersize=6, linewidth=1.5)
    ax.set_xticks(range(len(CONCEPT_ORDER)))
    ax.set_xticklabels(CONCEPT_ORDER, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{LANG_NAMES[lang]}  ({lang})", fontweight="bold",
                 color=FAMILY_COLOR.get(
                     df_lang[df_lang["lang"] == lang]["family"].iloc[0],
                     "black"))
    ax.grid(alpha=0.25)
    ax.set_ylim(0, 1.0)
    ax.axhline(0.5, color="grey", linewidth=0.5, ls="--")

axes[0].set_ylabel("Cluster-mask IoU vs English")
axes[1].legend(loc="lower right", frameon=True, fontsize=8)
plt.suptitle("Low-resource per-concept IoU profiles — base vs large CLIP\n"
             "Failure is concept-localised: same backbone, very different "
             "concept-level outcomes", y=1.04)
savefig("fig6_lowresource_concept_profile.png")


Saved: clip_multilingual_analysis/fig6_lowresource_concept_profile.png


In [11]:
# ============================================================
# 10) Energy comparison — dense CLIP grounding vs generative VLMs
# ============================================================
#
# Dense-CLIP figures come straight from the energy.json files; the
# generative-VLM column is reproduced from the parent paper (Table 1) with
# fixed Wh / 1K queries values for direct comparison. Both protocols use the
# same NVML 10 Hz sampler.

VLM_REFERENCE_WH_PER_1K = {
    "Phi-3-V":     66.3,
    "LLaVA-1.5":   82.5,
    "Qwen2-VL":   111.6,
    "LLaVA-1.6":  130.0,
    "InternVL2":  164.9,
}

ours = []
for _, r in df_energy.iterrows():
    ours.append((r["display"], r["wh_per_1k_queries"]))

all_rows = [(name, wh) for name, wh in VLM_REFERENCE_WH_PER_1K.items()] + ours
all_rows.sort(key=lambda x: x[1])

fig, ax = plt.subplots(figsize=(8.5, 4.2))
names = [r[0] for r in all_rows]
vals  = [r[1] for r in all_rows]
colors = ["#1f77b4" if n in [BACKBONES[k][0] for k in BACKBONES] else "#888888"
          for n in names]
bars = ax.barh(names, vals, color=colors, edgecolor="black", linewidth=0.5)
for b, v in zip(bars, vals):
    ax.text(b.get_width() * 1.01, b.get_y() + b.get_height() / 2,
            f"{v:.1f}", va="center", fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("Wh per 1,000 queries  (log scale, lower = more efficient)")
ax.set_title("Energy efficiency: dense-CLIP grounding vs generative VLM "
             "querying\n(same NVML 10 Hz protocol, parent-paper VLM "
             "figures from Table 1)")
ax.grid(axis="x", which="both", alpha=0.25)

# Annotate the speedup factor
min_clip = min(r[1] for r in ours)
max_vlm  = max(VLM_REFERENCE_WH_PER_1K.values())
min_vlm  = min(VLM_REFERENCE_WH_PER_1K.values())
ax.text(0.98, 0.05,
        f"Dense-CLIP grounding is\n"
        f"{min_vlm / min_clip:.1f}–{max_vlm / min_clip:.1f}× more "
        f"energy-efficient\n"
        f"than autoregressive VLM querying",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="grey"))

savefig("fig7_energy_comparison_vs_vlms.png")

print("Energy comparison (Wh per 1K queries):\n")
print(f"  {'Method':30s}  {'Wh/1K':>8s}")
print("  " + "-" * 42)
for n, v in all_rows:
    is_ours = n in [BACKBONES[k][0] for k in BACKBONES]
    marker = " *" if is_ours else "  "
    print(f"  {marker}{n:28s}  {v:>8.2f}")
print()
print(f"Energy savings vs cheapest VLM (Phi-3-V, 66.3 Wh/1K): "
      f"{VLM_REFERENCE_WH_PER_1K['Phi-3-V'] / min_clip:.1f}×")
print(f"Energy savings vs costliest VLM (InternVL2, 164.9 Wh/1K): "
      f"{VLM_REFERENCE_WH_PER_1K['InternVL2'] / min_clip:.1f}×")


Saved: clip_multilingual_analysis/fig7_energy_comparison_vs_vlms.png
Energy comparison (Wh per 1K queries):

  Method                             Wh/1K
  ------------------------------------------
   *XLM-R large + ViT-H/14            3.36
   *XLM-R base + ViT-B/32             3.87
    Phi-3-V                          66.30
    LLaVA-1.5                        82.50
    Qwen2-VL                        111.60
    LLaVA-1.6                       130.00
    InternVL2                       164.90

Energy savings vs cheapest VLM (Phi-3-V, 66.3 Wh/1K): 19.7×
Energy savings vs costliest VLM (InternVL2, 164.9 Wh/1K): 49.1×


In [16]:
# ============================================================
# 10) Energy comparison — dense CLIP grounding vs generative VLMs
# ============================================================
#
# Dense-CLIP figures come straight from the energy.json files; the
# generative-VLM column is reproduced from the parent paper (Table 1) with
# fixed Wh / 1K queries values for direct comparison. Both protocols use the
# same NVML 10 Hz sampler.

VLM_REFERENCE_WH_PER_1K = {
    "Phi-3-V":     66.3,
    "LLaVA-1.5":   82.5,
    "Qwen2-VL":   111.6,
    "LLaVA-1.6":  130.0,
    "InternVL2":  164.9,
}

ours = []
for _, r in df_energy.iterrows():
    ours.append((r["display"], r["wh_per_1k_queries"]))

all_rows = [(name, wh) for name, wh in VLM_REFERENCE_WH_PER_1K.items()] + ours
all_rows.sort(key=lambda x: x[1])

fig, ax = plt.subplots(figsize=(8.5, 4.6))
names = [r[0] for r in all_rows]
vals  = [r[1] for r in all_rows]
colors = ["#1f77b4" if n in [BACKBONES[k][0] for k in BACKBONES] else "#888888"
          for n in names]
bars = ax.barh(names, vals, color=colors, edgecolor="black", linewidth=0.5)

ax.set_xscale("log")

# Set explicit xlim so log-space label positioning is consistent.
xlim_min = 2.0
ax.set_xlim(xlim_min, max(vals) * 1.15)

# Place each value label INSIDE its bar at ~78% of the log-space distance
# from the left edge to the bar's right end. This keeps the label visually
# centred-right inside every bar regardless of magnitude (works for the
# small CLIP bars and the large InternVL2 bar alike).
for b, v in zip(bars, vals):
    log_min = np.log10(xlim_min)
    log_v   = np.log10(v)
    label_x = 10 ** (log_min + 0.78 * (log_v - log_min))
    ax.text(label_x, b.get_y() + b.get_height() / 2,
            f"{v:.1f}", va="center", ha="center",
            fontsize=9, color="white", fontweight="bold")

ax.set_xlabel("Wh per 1,000 queries  (log scale, lower = more efficient)")
ax.set_title("Energy efficiency: dense-CLIP grounding vs generative VLM "
             "querying\n(same NVML 10 Hz protocol, parent-paper VLM "
             "figures from Table 1)")
ax.grid(axis="x", which="both", alpha=0.25)

# Annotate the speedup factor in the empty whitespace below Phi-3-V
min_clip = min(r[1] for r in ours)
max_vlm  = max(VLM_REFERENCE_WH_PER_1K.values())
min_vlm  = min(VLM_REFERENCE_WH_PER_1K.values())
ax.text(0.97, 0.32,
        f"Dense-CLIP grounding is\n"
        f"{min_vlm / min_clip:.1f}\u2013{max_vlm / min_clip:.1f}\u00d7 more "
        f"energy-efficient\nthan autoregressive VLM querying",
        transform=ax.transAxes, ha="right", va="center",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="grey"))

savefig("fig7_energy_comparison_vs_vlms_o.png")

print("Energy comparison (Wh per 1K queries):\n")
print(f"  {'Method':30s}  {'Wh/1K':>8s}")
print("  " + "-" * 42)
for n, v in all_rows:
    is_ours = n in [BACKBONES[k][0] for k in BACKBONES]
    marker = " *" if is_ours else "  "
    print(f"  {marker}{n:28s}  {v:>8.2f}")
print()
print(f"Energy savings vs cheapest VLM (Phi-3-V, 66.3 Wh/1K): "
      f"{VLM_REFERENCE_WH_PER_1K['Phi-3-V'] / min_clip:.1f}x")
print(f"Energy savings vs costliest VLM (InternVL2, 164.9 Wh/1K): "
      f"{VLM_REFERENCE_WH_PER_1K['InternVL2'] / min_clip:.1f}x")

Saved: clip_multilingual_analysis/fig7_energy_comparison_vs_vlms_o.png
Energy comparison (Wh per 1K queries):

  Method                             Wh/1K
  ------------------------------------------
   *XLM-R large + ViT-H/14            3.36
   *XLM-R base + ViT-B/32             3.87
    Phi-3-V                          66.30
    LLaVA-1.5                        82.50
    Qwen2-VL                        111.60
    LLaVA-1.6                       130.00
    InternVL2                       164.90

Energy savings vs cheapest VLM (Phi-3-V, 66.3 Wh/1K): 19.7x
Energy savings vs costliest VLM (InternVL2, 164.9 Wh/1K): 49.1x


In [17]:
# ============================================================
# 10) Energy comparison — dense CLIP grounding vs generative VLMs
# ============================================================
#
# Dense-CLIP figures come straight from the energy.json files; the
# generative-VLM column is reproduced from the parent paper (Table 1) with
# fixed Wh / 1K queries values for direct comparison. Both protocols use the
# same NVML 10 Hz sampler.

VLM_REFERENCE_WH_PER_1K = {
    "Phi-3-V":     66.3,
    "LLaVA-1.5":   82.5,
    "Qwen2-VL":   111.6,
    "LLaVA-1.6":  130.0,
    "InternVL2":  164.9,
}

ours = []
for _, r in df_energy.iterrows():
    ours.append((r["display"], r["wh_per_1k_queries"]))

all_rows = [(name, wh) for name, wh in VLM_REFERENCE_WH_PER_1K.items()] + ours
all_rows.sort(key=lambda x: x[1])

fig, ax = plt.subplots(figsize=(10, 4.2))
names = [r[0] for r in all_rows]
vals  = [r[1] for r in all_rows]
colors = ["#1f77b4" if n in [BACKBONES[k][0] for k in BACKBONES] else "#888888"
          for n in names]
bars = ax.barh(names, vals, color=colors, edgecolor="black", linewidth=0.5)
for b, v in zip(bars, vals):
    ax.text(b.get_width() * 1.01, b.get_y() + b.get_height() / 2,
            f"{v:.1f}", va="center", fontsize=9)

ax.set_xscale("log")
ax.set_xlim(right=max(vals) * 1.4)
ax.set_xlabel("Wh per 1,000 queries  (log scale, lower = more efficient)")
ax.set_title("Energy efficiency: dense-CLIP grounding vs generative VLM "
             "querying\n(same NVML 10 Hz protocol, parent-paper VLM "
             "figures from Table 1)")
ax.grid(axis="x", which="both", alpha=0.25)

# Annotate the speedup factor
min_clip = min(r[1] for r in ours)
max_vlm  = max(VLM_REFERENCE_WH_PER_1K.values())
min_vlm  = min(VLM_REFERENCE_WH_PER_1K.values())
ax.text(0.98, 0.05,
        f"Dense-CLIP grounding is\n"
        f"{min_vlm / min_clip:.1f}\u2013{max_vlm / min_clip:.1f}\u00d7 more "
        f"energy-efficient\n"
        f"than autoregressive VLM querying",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="grey"))

savefig("fig7_energy_comparison_vs_vlms_o2.png")

print("Energy comparison (Wh per 1K queries):\n")
print(f"  {'Method':30s}  {'Wh/1K':>8s}")
print("  " + "-" * 42)
for n, v in all_rows:
    is_ours = n in [BACKBONES[k][0] for k in BACKBONES]
    marker = " *" if is_ours else "  "
    print(f"  {marker}{n:28s}  {v:>8.2f}")
print()
print(f"Energy savings vs cheapest VLM (Phi-3-V, 66.3 Wh/1K): "
      f"{VLM_REFERENCE_WH_PER_1K['Phi-3-V'] / min_clip:.1f}x")
print(f"Energy savings vs costliest VLM (InternVL2, 164.9 Wh/1K): "
      f"{VLM_REFERENCE_WH_PER_1K['InternVL2'] / min_clip:.1f}x")

Saved: clip_multilingual_analysis/fig7_energy_comparison_vs_vlms_o2.png
Energy comparison (Wh per 1K queries):

  Method                             Wh/1K
  ------------------------------------------
   *XLM-R large + ViT-H/14            3.36
   *XLM-R base + ViT-B/32             3.87
    Phi-3-V                          66.30
    LLaVA-1.5                        82.50
    Qwen2-VL                        111.60
    LLaVA-1.6                       130.00
    InternVL2                       164.90

Energy savings vs cheapest VLM (Phi-3-V, 66.3 Wh/1K): 19.7x
Energy savings vs costliest VLM (InternVL2, 164.9 Wh/1K): 49.1x


In [12]:
# ============================================================
# 11) Per-concept HR vs LR gap — diagnostic of where the failure concentrates
# ============================================================

rows = []
for backbone in BACKBONES:
    for concept in CONCEPT_ORDER:
        sub = df_conc[(df_conc["backbone"] == backbone) &
                      (df_conc["concept"] == concept) &
                      (df_conc["lang"] != "en")]
        hr = sub[~sub["low_resource"]]["iou_cluster_mask"].mean()
        lr = sub[ sub["low_resource"]]["iou_cluster_mask"].mean()
        rows.append({"backbone": backbone, "concept": concept,
                     "hr": hr, "lr": lr, "gap": hr - lr})
df_concept_gap = pd.DataFrame(rows)
df_concept_gap.to_csv(os.path.join(OUTDIR, "table_concept_gap.csv"), index=False)

# Plot side-by-side: gap by concept for both backbones
fig, ax = plt.subplots(figsize=(11, 4.4))
pivot = df_concept_gap.pivot_table(index="concept", columns="backbone",
                                   values="gap").reindex(CONCEPT_ORDER)

x = np.arange(len(pivot))
bar_w = 0.4
for i, bb in enumerate(BACKBONES):
    ax.bar(x + (i - 0.5) * bar_w, pivot[bb].values, width=bar_w,
           color=BACKBONE_COLOR[bb], edgecolor="black", linewidth=0.5,
           label=BACKBONES[bb][0])

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("HR − LR cluster-mask IoU gap")
ax.set_title("Per-concept HR vs LR gap, both backbones\n"
             "Larger bar = stronger language disparity for that concept")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper left", frameon=True)
ax.axhline(0, color="black", linewidth=0.7)

savefig("fig8_concept_gap_both_backbones.png")

print("Top 5 concepts with widest HR−LR gap (large backbone):")
top = (df_concept_gap[df_concept_gap["backbone"] == "dense_clip_xlmr_large"]
            .nlargest(5, "gap"))
display(top.round(4).to_string(index=False))
print()
print("Bottom 5 concepts (smallest gap = most language-robust):")
bot = (df_concept_gap[df_concept_gap["backbone"] == "dense_clip_xlmr_large"]
            .nsmallest(5, "gap"))
display(bot.round(4).to_string(index=False))


Saved: clip_multilingual_analysis/fig8_concept_gap_both_backbones.png
Top 5 concepts with widest HR−LR gap (large backbone):


'             backbone       concept     hr     lr    gap\ndense_clip_xlmr_large          road 0.6796 0.3374 0.3422\ndense_clip_xlmr_large  traffic_sign 0.6811 0.4370 0.2442\ndense_clip_xlmr_large    pedestrian 0.5693 0.3492 0.2201\ndense_clip_xlmr_large        person 0.4948 0.3483 0.1465\ndense_clip_xlmr_large traffic_light 0.6659 0.5252 0.1408'


Bottom 5 concepts (smallest gap = most language-robust):


'             backbone    concept     hr     lr    gap\ndense_clip_xlmr_large        bus 0.6266 0.5870 0.0397\ndense_clip_xlmr_large motorcycle 0.5749 0.5167 0.0581\ndense_clip_xlmr_large        car 0.6528 0.5906 0.0622\ndense_clip_xlmr_large      truck 0.6361 0.5500 0.0861\ndense_clip_xlmr_large    bicycle 0.5311 0.4397 0.0915'

In [13]:
# ============================================================
# 12) Headline summary table for the paper
# ============================================================

summary_rows = []
for backbone in BACKBONES:
    sub_lang = df_lang[df_lang["backbone"] == backbone]
    sub_pair = df_pair[df_pair["backbone"] == backbone]
    sub_eng  = df_energy[df_energy["backbone"] == backbone].iloc[0]

    # HR vs LR aggregates
    non_en = sub_lang[sub_lang["lang"] != "en"]
    hr = non_en[~non_en["low_resource"]]
    lr = non_en[ non_en["low_resource"]]

    # Friedman / Wilcoxon already computed in df_stats
    fr   = df_stats[(df_stats["backbone"] == backbone) &
                    (df_stats["metric"] == "iou_cluster_mask")].iloc[0]

    summary_rows.append({
        "Backbone":            BACKBONES[backbone][0],
        "Visual params (M)":   BACKBONES[backbone][1],
        "n paired obs":        f"{int(fr['wilcoxon_n']):,}",
        "Mean IoU (HR)":       f"{hr['iou_cluster_mask'].mean():.4f}",
        "Mean IoU (LR)":       f"{lr['iou_cluster_mask'].mean():.4f}",
        "HR-LR IoU gap":       f"{(hr['iou_cluster_mask'].mean() - lr['iou_cluster_mask'].mean()):+.4f}",
        "Wilcoxon HR>LR p":    (f"{fr['wilcoxon_p']:.2e}"
                                if fr['wilcoxon_p'] > 0
                                else "<1e-300"),
        "Wh / 1K queries":     f"{sub_eng['wh_per_1k_queries']:.3f}",
        "Avg watts":           f"{sub_eng['avg_watts']:.1f}",
        "Total Wh":            f"{sub_eng['total_wh']:.2f}",
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(OUTDIR, "TABLE1_paper_summary.csv"),
                  index=False)

print("=" * 80)
print("HEADLINE TABLE FOR THE PAPER")
print("=" * 80)
display(df_summary.to_string(index=False))
print()
print("Saved:", os.path.join(OUTDIR, "TABLE1_paper_summary.csv"))


HEADLINE TABLE FOR THE PAPER


'              Backbone  Visual params (M) n paired obs Mean IoU (HR) Mean IoU (LR) HR-LR IoU gap Wilcoxon HR>LR p Wh / 1K queries Avg watts Total Wh\n XLM-R base + ViT-B/32                 87        2,310        0.6123        0.4978       +0.1145          <1e-300           3.868      98.9   116.15\nXLM-R large + ViT-H/14                632        2,310        0.6076        0.4647       +0.1428          <1e-300           3.358      99.3   100.85'


Saved: clip_multilingual_analysis/TABLE1_paper_summary.csv


In [14]:
# ============================================================
# 13) Write statistical report (paper draft material)
# ============================================================

report_path = os.path.join(OUTDIR, "CLIP_MULTILINGUAL_REPORT.txt")
lines = []
P = lines.append

P("=" * 78)
P("DENSE-CLIP MULTILINGUAL GROUNDING — STATISTICAL REPORT")
P("=" * 78)
P("")
P("Frozen subset: 210 BDD100K images")
P("Languages    : 13 (3 low-resource: ar, eu, lb)")
P("Concepts     : 11 BDD100K-relevant nouns")
P("Reference    : English (en)")
P("Paired obs.  : 13 langs × 11 concepts × 210 images = 30,030 records")
P("               (2,310 per non-EN language)")
P("")

# Per-backbone block
for backbone in BACKBONES:
    P("-" * 78)
    P(f"BACKBONE  : {BACKBONES[backbone][0]}")
    P(f"           ({backbone})")
    P("-" * 78)

    e = df_energy[df_energy["backbone"] == backbone].iloc[0]
    P(f"Energy    : {e['wh_per_1k_queries']:.3f} Wh per 1,000 queries")
    P(f"            {e['avg_watts']:.1f} W avg, {e['total_wh']:.2f} Wh total over "
      f"{e['duration_sec']:.0f} s")
    P("")

    # HR vs LR block
    sub_lang = df_lang[df_lang["backbone"] == backbone]
    non_en   = sub_lang[sub_lang["lang"] != "en"]
    hr       = non_en[~non_en["low_resource"]]
    lr       = non_en[ non_en["low_resource"]]
    P("Per-language IoU (mean cluster-mask IoU vs EN, n=2310 each):")
    for lang in [l for l in LANG_ORDER if l != "en"]:
        row = sub_lang[sub_lang["lang"] == lang].iloc[0]
        tag = " [LR]" if row["low_resource"] else ""
        P(f"  {row['lang_name']:18s} ({lang:5s}){tag}: "
          f"IoU = {row['iou_cluster_mask']:.4f}, "
          f"Spearman = {row['spearman']:.4f}, "
          f"peak ratio = {row['peak_ratio_lang_over_ref']:.4f}")
    P("")

    P(f"HR mean IoU: {hr['iou_cluster_mask'].mean():.4f} "
      f"(n={hr.shape[0]} languages)")
    P(f"LR mean IoU: {lr['iou_cluster_mask'].mean():.4f} "
      f"(n={lr.shape[0]} languages)")
    P(f"HR − LR gap: {hr['iou_cluster_mask'].mean() - lr['iou_cluster_mask'].mean():+.4f}")
    P("")

    # Statistical tests
    sub_stats = df_stats[df_stats["backbone"] == backbone]
    P("Statistical tests (paired by image × concept):")
    for _, r in sub_stats.iterrows():
        p_w = r['wilcoxon_p']
        p_w_str = f"{p_w:.2e}" if p_w > 0 else "<1e-300"
        p_f = r['friedman_p']
        p_f_str = f"{p_f:.2e}" if p_f > 0 else "<1e-300"
        P(f"  {r['metric']:30s}  "
          f"Friedman χ²={r['friedman_stat']:.1f} (p={p_f_str}, n={r['friedman_n']})  |  "
          f"Wilcoxon HR>LR Δ={r['wilcoxon_meandif']:+.4f} (p={p_w_str})")
    P("")

# Cross-backbone Δ block
P("-" * 78)
P("SCALE EFFECT: Δ IoU = IoU(large) − IoU(base)  per language")
P("-" * 78)
for _, r in deltas.iterrows():
    tag = " [LR]" if r["low_resource"] else ""
    P(f"  {r['lang_name']:18s} ({r['lang']:5s}){tag}: "
      f"base={r['iou_base']:.4f}, large={r['iou_large']:.4f}, "
      f"Δ={r['delta']:+.4f}")
P("")

P("Headline: low-resource languages show a NEGATIVE Δ IoU on average "
  f"(mean Δ = {deltas[deltas['low_resource']]['delta'].mean():+.4f}), "
  "while high-resource languages show a small positive or near-zero shift "
  f"(mean Δ = {deltas[~deltas['low_resource']]['delta'].mean():+.4f}). "
  "Scaling the multilingual encoder does not close the gap — it widens "
  "it for the structural failure cases (Basque, Luxembourgish).")
P("")

P("=" * 78)
report_text = "\n".join(lines)
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"Wrote: {report_path}")
print(f"Total length: {len(report_text)} characters")
print()
print("First 80 lines:")
print("\n".join(lines[:80]))


Wrote: clip_multilingual_analysis/CLIP_MULTILINGUAL_REPORT.txt
Total length: 6116 characters

First 80 lines:
DENSE-CLIP MULTILINGUAL GROUNDING — STATISTICAL REPORT

Frozen subset: 210 BDD100K images
Languages    : 13 (3 low-resource: ar, eu, lb)
Concepts     : 11 BDD100K-relevant nouns
Reference    : English (en)
Paired obs.  : 13 langs × 11 concepts × 210 images = 30,030 records
               (2,310 per non-EN language)

------------------------------------------------------------------------------
BACKBONE  : XLM-R base + ViT-B/32
           (dense_clip_xlmr_base)
------------------------------------------------------------------------------
Energy    : 3.868 Wh per 1,000 queries
            98.9 W avg, 116.15 Wh total over 4227 s

Per-language IoU (mean cluster-mask IoU vs EN, n=2310 each):
  Spanish            (es   ): IoU = 0.6535, Spearman = 0.9121, peak ratio = 0.9680
  French             (fr   ): IoU = 0.6450, Spearman = 0.8997, peak ratio = 0.9656
  Italian            (it   

In [15]:
# ============================================================
# 14) Package & download all outputs
# ============================================================

import zipfile
zip_path = os.path.join(OUTDIR, "clip_multilingual_analysis_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUTDIR):
        for fn in files:
            if fn.endswith(".zip"):
                continue
            p = os.path.join(root, fn)
            z.write(p, arcname=os.path.relpath(p, OUTDIR))

print(f"Wrote: {zip_path}")
print()
print("Contents:")
for root, _, files in os.walk(OUTDIR):
    for fn in sorted(files):
        if fn.endswith(".zip"):
            continue
        p = os.path.join(root, fn)
        size_kb = os.path.getsize(p) / 1024
        print(f"  {os.path.relpath(p, OUTDIR):60s}  {size_kb:6.1f} KB")


Wrote: clip_multilingual_analysis/clip_multilingual_analysis_outputs.zip

Contents:
  CLIP_MULTILINGUAL_REPORT.txt                                     6.0 KB
  TABLE1_paper_summary.csv                                         0.3 KB
  df_conc.csv                                                    107.6 KB
  df_energy.csv                                                    0.4 KB
  df_lang.csv                                                     10.1 KB
  fig1_per_language_iou_both_backbones.png                       147.7 KB
  fig2_hr_vs_lr_forest.png                                       120.1 KB
  fig3_delta_iou_scale.png                                       165.4 KB
  fig4_signal_vs_spatial.png                                     180.2 KB
  fig5_concept_heatmaps_both_backbones.png                       344.5 KB
  fig6_lowresource_concept_profile.png                           231.7 KB
  fig7_energy_comparison_vs_vlms.png                             110.0 KB
  fig8_concept_gap_both_back